In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi

# Crypto Portfolio FTMO Strategy
---

## Strategy Overview
Multi-asset crypto portfolio combining **MACD Crossover** and **Donchian Breakout** strategies
across the top-performing crypto assets from our backtest research.

### Portfolio Allocation (based on OOS performance analysis)

| Strategy | Asset | Weight | OOS Sharpe | Rationale |
|----------|-------|--------|-----------|----------|
| Donchian Breakout | XRP-USD | 30% | 1.48 | Best OOS performer overall |
| MACD Crossover | BTC-USD | 20% | 0.90 | Most consistent IS-to-OOS |
| MACD Crossover | XRP-USD | 20% | 0.88 | Diversified signal timing |
| MACD Crossover | ETH-USD | 15% | 0.69 | Large-cap diversification |
| Donchian Breakout | BTC-USD | 15% | 0.55 | Low-frequency trend overlay |

### FTMO Rules
- Account: $100,000 | Profit Target: 10% | Max Daily Loss: 5% | Max Total Loss: 10%
- Crypto leverage: 1:3 (standard), 1:1 (swing)

### Optimal Parameters (from grid search)
- MACD BTC: fast=30, slow=59, signal=6
- MACD XRP: fast=22, slow=39, signal=3
- MACD ETH: fast=29, slow=44, signal=5
- Donchian XRP: entry=10, exit=3, filter=20
- Donchian BTC: entry=31, exit=17, filter=70

In [ ]:
# CONFIGURATION

START_DATE = '2018-01-01'
TRAIN_RATIO = 0.60
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
FREQ = 'D'

# Portfolio components: (strategy, ticker, weight, params)
PORTFOLIO = [
    ('donchian', 'XRP-USD', 0.30, {'entry': 10, 'exit': 3, 'filter': 20}),
    ('macd',     'BTC-USD', 0.20, {'fast': 30, 'slow': 59, 'signal': 6}),
    ('macd',     'XRP-USD', 0.20, {'fast': 22, 'slow': 39, 'signal': 3}),
    ('macd',     'ETH-USD', 0.15, {'fast': 29, 'slow': 44, 'signal': 5}),
    ('donchian', 'BTC-USD', 0.15, {'entry': 31, 'exit': 17, 'filter': 70}),
]

TICKERS = list(set(t for _, t, _, _ in PORTFOLIO))
print(f"Portfolio: {len(PORTFOLIO)} components across {len(TICKERS)} assets")
print(f"Tickers: {TICKERS}")
for strat, tick, w, p in PORTFOLIO:
    print(f"  {strat.upper():10s} {tick:10s} {w:.0%}  params={p}")

In [ ]:
# DOWNLOAD DATA FOR ALL TICKERS

data = {}
for ticker in TICKERS:
    df = download(ticker, START_DATE)
    if isinstance(df.columns, pd.MultiIndex):
        close = df[('Close', ticker)].astype(float).squeeze()
        high = df[('High', ticker)].astype(float).squeeze()
        low = df[('Low', ticker)].astype(float).squeeze()
    else:
        close = df['Close'].astype(float).squeeze()
        high = df['High'].astype(float).squeeze()
        low = df['Low'].astype(float).squeeze()
    close.name = 'price'
    data[ticker] = {'close': close, 'high': high, 'low': low, 'df': df}
    print(f"{ticker}: {len(close)} bars from {close.index[0].date()} to {close.index[-1].date()}")

# Align all series to common date range
common_start = max(d['close'].index[0] for d in data.values())
common_end = min(d['close'].index[-1] for d in data.values())
for ticker in TICKERS:
    mask = (data[ticker]['close'].index >= common_start) & (data[ticker]['close'].index <= common_end)
    data[ticker]['close'] = data[ticker]['close'][mask]
    data[ticker]['high'] = data[ticker]['high'][mask]
    data[ticker]['low'] = data[ticker]['low'][mask]

n_bars = len(data[TICKERS[0]]['close'])
split_idx = int(n_bars * TRAIN_RATIO)
print(f"\nCommon range: {common_start.date()} to {common_end.date()} ({n_bars} bars)")
print(f"Train: {n_bars * TRAIN_RATIO:.0f} bars | Val: {n_bars * (1-TRAIN_RATIO):.0f} bars")

In [ ]:
# SIGNAL GENERATION FUNCTIONS

def macd_signals(close, fast, slow, signal):
    """Generate MACD crossover signals with 1-bar execution delay."""
    ml, sl, _ = talib.MACD(close.values.astype(float),
                           fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_s = pd.Series(ml, index=close.index)
    sig_s = pd.Series(sl, index=close.index)
    
    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)),
                        index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)),
                      index=close.index, dtype=bool)
    return entries, exits


def donchian_signals(close, high, low, entry_period, exit_period, filter_period):
    """Generate Donchian breakout signals with 1-bar execution delay."""
    upper_channel = pd.Series(talib.MAX(high.values, entry_period), index=close.index).shift(1)
    lower_channel = pd.Series(talib.MIN(low.values, exit_period), index=close.index).shift(1)
    trend_filter = pd.Series(talib.SMA(close.values, filter_period), index=close.index).shift(1)
    
    entries_raw = (close > upper_channel) & (close > trend_filter)
    exits_raw = (close < lower_channel)
    
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)),
                        index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)),
                      index=close.index, dtype=bool)
    return entries, exits

print("Signal functions defined.")

In [ ]:
# RUN EACH COMPONENT STRATEGY & COLLECT DAILY RETURNS

component_results = []
component_daily_returns = []
component_portfolios = []

for strat, ticker, weight, params in PORTFOLIO:
    close = data[ticker]['close']
    high = data[ticker]['high']
    low = data[ticker]['low']
    
    if strat == 'macd':
        entries, exits = macd_signals(close, params['fast'], params['slow'], params['signal'])
        label = f"MACD({params['fast']},{params['slow']},{params['signal']})"
    elif strat == 'donchian':
        entries, exits = donchian_signals(close, high, low, params['entry'], params['exit'], params['filter'])
        label = f"Donchian({params['entry']},{params['exit']},{params['filter']})"
    
    # Full-sample portfolio
    pf = vbt.Portfolio.from_signals(
        close=close, entries=entries, exits=exits,
        init_cash=INIT_CASH * weight, fees=FEES, slippage=SLIPPAGE, freq=FREQ
    )
    
    # IS / OOS split
    train_close = close.iloc[:split_idx]
    val_close = close.iloc[split_idx:]
    
    pf_is = vbt.Portfolio.from_signals(
        close=train_close, entries=entries.iloc[:split_idx], exits=exits.iloc[:split_idx],
        init_cash=INIT_CASH * weight, fees=FEES, slippage=SLIPPAGE, freq=FREQ
    )
    pf_oos = vbt.Portfolio.from_signals(
        close=val_close, entries=entries.iloc[split_idx:], exits=exits.iloc[split_idx:],
        init_cash=INIT_CASH * weight, fees=FEES, slippage=SLIPPAGE, freq=FREQ
    )
    
    # Collect metrics
    trades_obj = pf.trades
    tr = trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else np.array(trades_obj.returns)
    pos = tr[tr > 0]
    neg = tr[tr < 0]
    win_rate = (len(pos) / len(tr) * 100) if len(tr) > 0 else 0
    pf_ratio = pos.sum() / abs(neg.sum()) if len(neg) > 0 and abs(neg.sum()) > 0 else np.inf
    
    result = {
        'component': f"{strat.upper()} {ticker}",
        'label': label,
        'weight': weight,
        'full_sharpe': float(pf.sharpe_ratio(freq=FREQ)),
        'full_return': float(pf.total_return()),
        'full_maxdd': float(pf.max_drawdown()),
        'is_sharpe': float(pf_is.sharpe_ratio(freq=FREQ)),
        'oos_sharpe': float(pf_oos.sharpe_ratio(freq=FREQ)),
        'is_return': float(pf_is.total_return()),
        'oos_return': float(pf_oos.total_return()),
        'oos_maxdd': float(pf_oos.max_drawdown()),
        'trades': len(pf.trades),
        'win_rate': win_rate,
        'profit_factor': pf_ratio,
    }
    component_results.append(result)
    component_daily_returns.append(pf.returns().rename(f"{strat}_{ticker}"))
    component_portfolios.append(pf)
    
    print(f"{result['component']:25s} | Sharpe IS={result['is_sharpe']:.2f} OOS={result['oos_sharpe']:.2f} | "
          f"Return OOS={result['oos_return']:.1%} | DD={result['oos_maxdd']:.1%} | "
          f"Trades={result['trades']} | WR={result['win_rate']:.1f}%")

results_df = pd.DataFrame(component_results)
print(f"\n{len(component_results)} components evaluated.")

In [ ]:
# PORTFOLIO COMBINATION — WEIGHTED DAILY RETURNS

# Combine weighted daily returns
all_rets = pd.DataFrame(component_daily_returns).T
all_rets = all_rets.fillna(0)

# Weighted portfolio return
weights = np.array([w for _, _, w, _ in PORTFOLIO])
portfolio_daily_returns = (all_rets * weights).sum(axis=1)

# Build portfolio equity curve
portfolio_equity = INIT_CASH * (1 + portfolio_daily_returns).cumprod()

# Split IS/OOS
port_eq_is = portfolio_equity.iloc[:split_idx]
port_eq_oos = portfolio_equity.iloc[split_idx:]
port_rets_is = portfolio_daily_returns.iloc[:split_idx]
port_rets_oos = portfolio_daily_returns.iloc[split_idx:]

# Portfolio metrics
def compute_metrics(daily_rets, label):
    total_ret = (1 + daily_rets).prod() - 1
    n_years = len(daily_rets) / 252
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1 if n_years > 0 else 0
    ann_vol = daily_rets.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cum = (1 + daily_rets).cumprod()
    dd = cum / cum.cummax() - 1
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 1e-9 else np.nan
    sortino_denom = daily_rets[daily_rets < 0].std() * np.sqrt(252)
    sortino = ann_ret / sortino_denom if sortino_denom > 0 else np.nan
    return {
        'label': label, 'total_return': total_ret, 'ann_return': ann_ret,
        'ann_vol': ann_vol, 'sharpe': sharpe, 'sortino': sortino,
        'max_dd': max_dd, 'calmar': calmar
    }

m_full = compute_metrics(portfolio_daily_returns, 'Full')
m_is = compute_metrics(port_rets_is, 'IS')
m_oos = compute_metrics(port_rets_oos, 'OOS')

print("=" * 70)
print("CRYPTO PORTFOLIO — COMBINED METRICS")
print("=" * 70)
print(f"{'Metric':<28} {'Full':>10} {'IS':>10} {'OOS':>10}")
print("-" * 58)
print(f"{'Total Return':<28} {m_full['total_return']:>9.1%} {m_is['total_return']:>9.1%} {m_oos['total_return']:>9.1%}")
print(f"{'Ann. Return':<28} {m_full['ann_return']:>9.1%} {m_is['ann_return']:>9.1%} {m_oos['ann_return']:>9.1%}")
print(f"{'Ann. Volatility':<28} {m_full['ann_vol']:>9.1%} {m_is['ann_vol']:>9.1%} {m_oos['ann_vol']:>9.1%}")
print(f"{'Sharpe Ratio':<28} {m_full['sharpe']:>10.3f} {m_is['sharpe']:>10.3f} {m_oos['sharpe']:>10.3f}")
print(f"{'Sortino Ratio':<28} {m_full['sortino']:>10.3f} {m_is['sortino']:>10.3f} {m_oos['sortino']:>10.3f}")
print(f"{'Max Drawdown':<28} {m_full['max_dd']:>9.1%} {m_is['max_dd']:>9.1%} {m_oos['max_dd']:>9.1%}")
print(f"{'Calmar Ratio':<28} {m_full['calmar']:>10.3f} {m_is['calmar']:>10.3f} {m_oos['calmar']:>10.3f}")
print("=" * 70)

In [ ]:
# CORRELATION MATRIX — COMPONENT DIVERSIFICATION

corr = all_rets.corr()
print("Component Return Correlations:")
print(corr.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Correlation heatmap
im = axes[0].imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(corr.columns)))
axes[0].set_yticks(range(len(corr.columns)))
axes[0].set_xticklabels([c.replace('_', '\n') for c in corr.columns], fontsize=8, rotation=45, ha='right')
axes[0].set_yticklabels([c.replace('_', '\n') for c in corr.columns], fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[0].text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=9)
axes[0].set_title('Component Return Correlations', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Individual equity curves
for i, (strat, ticker, weight, params) in enumerate(PORTFOLIO):
    eq = component_portfolios[i].value()
    axes[1].plot(eq.index, eq.values, linewidth=1.5, label=f'{strat.upper()} {ticker} ({weight:.0%})')
axes[1].plot(portfolio_equity.index, portfolio_equity.values, linewidth=2.5, color='black', label='Combined Portfolio')
axes[1].axvline(x=portfolio_equity.index[split_idx], color='red', linestyle=':', alpha=0.6, label='IS/OOS Split')
axes[1].set_title('Component vs Combined Equity Curves', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Portfolio Value ($)', fontsize=11)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend(fontsize=8, loc='upper left')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

avg_corr = corr.values[np.triu_indices(len(corr), k=1)].mean()
print(f"\nAverage pairwise correlation: {avg_corr:.3f}")
print(f"Diversification benefit: {'GOOD' if avg_corr < 0.5 else 'MODERATE' if avg_corr < 0.7 else 'LOW'}")

In [ ]:
# PORTFOLIO EQUITY CURVE + BUY & HOLD COMPARISON

# Buy & hold = equal-weight crypto basket
bh_rets_list = []
for ticker in TICKERS:
    c = data[ticker]['close']
    bh_rets_list.append(c.pct_change().fillna(0).rename(ticker))

bh_df = pd.DataFrame(bh_rets_list).T.fillna(0)
# Weight B&H by same ticker weights (sum weights per ticker)
ticker_weights = {}
for _, t, w, _ in PORTFOLIO:
    ticker_weights[t] = ticker_weights.get(t, 0) + w
bh_weights = np.array([ticker_weights[t] for t in bh_df.columns])
bh_port_rets = (bh_df * bh_weights).sum(axis=1)
bh_equity = INIT_CASH * (1 + bh_port_rets).cumprod()

m_bh = compute_metrics(bh_port_rets, 'B&H')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

# Equity curves
ax1.plot(portfolio_equity.index[:split_idx], portfolio_equity.values[:split_idx],
         color='#1f77b4', linewidth=1.8, label='Strategy (IS)', alpha=0.9)
ax1.plot(portfolio_equity.index[split_idx:], portfolio_equity.values[split_idx:],
         color='#ff7f0e', linewidth=1.8, label='Strategy (OOS)', alpha=0.9)
ax1.plot(bh_equity.index, bh_equity.values, color='gray', linewidth=1.2, label='Buy & Hold', alpha=0.5, linestyle='--')
ax1.axvline(x=portfolio_equity.index[split_idx], color='red', linestyle=':', alpha=0.6, label='Train/Val Split')

stats_text = (
    f"CRYPTO PORTFOLIO (5 components)\n"
    f"{'='*30}\n"
    f"Total Return:    {m_full['total_return']:.1%}\n"
    f"Sharpe (Full):   {m_full['sharpe']:.3f}\n"
    f"Sharpe (OOS):    {m_oos['sharpe']:.3f}\n"
    f"Max Drawdown:    {m_full['max_dd']:.1%}\n"
    f"Calmar Ratio:    {m_full['calmar']:.3f}\n"
    f"{'='*30}\n"
    f"B&H Return:      {m_bh['total_return']:.1%}\n"
    f"B&H Sharpe:      {m_bh['sharpe']:.3f}\n"
    f"B&H Max DD:      {m_bh['max_dd']:.1%}"
)
props = dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='#333333', alpha=0.92)
ax1.text(0.015, 0.97, stats_text, transform=ax1.transAxes, fontsize=9.5,
         verticalalignment='top', fontfamily='monospace', bbox=props)
ax1.set_title('Crypto Portfolio vs Buy & Hold', fontsize=15, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.25)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Drawdown
cum = (1 + portfolio_daily_returns).cumprod()
dd = (cum / cum.cummax() - 1) * 100
ax2.fill_between(dd.index, dd.values, 0, color='#d62728', alpha=0.5, label='Strategy DD')
cum_bh = (1 + bh_port_rets).cumprod()
dd_bh = (cum_bh / cum_bh.cummax() - 1) * 100
ax2.plot(dd_bh.index, dd_bh.values, color='gray', linewidth=0.8, alpha=0.6, linestyle='--', label='B&H DD')
ax2.axvline(x=portfolio_equity.index[split_idx], color='red', linestyle=':', alpha=0.5)
ax2.set_title('Underwater Plot (Drawdown %)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Drawdown %', fontsize=11)
ax2.legend(loc='lower right', fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# MONTE CARLO SIMULATION — FTMO CHALLENGE PASS PROBABILITY

print("=" * 70)
print("MONTE CARLO SIMULATION -- FTMO CHALLENGE FEASIBILITY")
print("=" * 70)

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30
N_SIMULATIONS = 10_000

daily_rets = portfolio_daily_returns.values
daily_rets = daily_rets[~np.isnan(daily_rets)]

print(f"\nPortfolio daily returns: {len(daily_rets)} days")
print(f"  Mean daily return:    {np.mean(daily_rets)*100:.4f}%")
print(f"  Std daily return:     {np.std(daily_rets)*100:.4f}%")
print(f"  Skewness:             {float(stats.skew(daily_rets)):.3f}")
print(f"  Kurtosis:             {float(stats.kurtosis(daily_rets)):.3f}")
print(f"\nFTMO Rules: ${FTMO_ACCOUNT:,.0f} | Target: {PROFIT_TARGET:.0%} | Daily DD: {MAX_DAILY_LOSS:.0%} | Total DD: {MAX_TOTAL_LOSS:.0%}")
print(f"Simulations: {N_SIMULATIONS:,} | Window: {TRADING_DAYS} days")

np.random.seed(42)
equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
equity_paths[:, 0] = FTMO_ACCOUNT

passed = np.zeros(N_SIMULATIONS, dtype=bool)
blown = np.zeros(N_SIMULATIONS, dtype=bool)
daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
final_equity = np.zeros(N_SIMULATIONS)

for sim in range(N_SIMULATIONS):
    sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
    eq = FTMO_ACCOUNT
    sim_passed = sim_blown_total = sim_blown_daily = False

    for day in range(TRADING_DAYS):
        day_start_eq = eq
        eq = eq * (1 + sim_rets[day])
        equity_paths[sim, day + 1] = eq

        daily_loss = (eq - day_start_eq) / FTMO_ACCOUNT
        if daily_loss < -MAX_DAILY_LOSS:
            sim_blown_daily = True
            equity_paths[sim, day + 2:] = eq
            break

        total_loss = (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT
        if total_loss < -MAX_TOTAL_LOSS:
            sim_blown_total = True
            equity_paths[sim, day + 2:] = eq
            break

        if total_loss >= PROFIT_TARGET:
            sim_passed = True
            equity_paths[sim, day + 2:] = eq
            break

    final_equity[sim] = equity_paths[sim, -1]
    passed[sim] = sim_passed
    blown[sim] = sim_blown_total
    daily_blown[sim] = sim_blown_daily

n_passed = passed.sum()
n_blown_total = blown.sum()
n_blown_daily = daily_blown.sum()
n_survived = N_SIMULATIONS - n_blown_total - n_blown_daily - n_passed

print(f"\n{'=' * 70}")
print(f"MONTE CARLO RESULTS ({N_SIMULATIONS:,} simulations)")
print(f"{'=' * 70}")
print(f"  PASSED (hit {PROFIT_TARGET:.0%} target):    {n_passed:>6,} ({n_passed/N_SIMULATIONS*100:.1f}%)")
print(f"  BLOWN (max total loss):       {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)")
print(f"  BLOWN (max daily loss):       {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)")
print(f"  Still trading (no trigger):   {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)")
print(f"  Median final equity:         ${np.median(final_equity):>10,.0f}")
print(f"  Mean final equity:           ${np.mean(final_equity):>10,.0f}")
print(f"  5th pctl equity:             ${np.percentile(final_equity, 5):>10,.0f}")
print(f"  95th pctl equity:            ${np.percentile(final_equity, 95):>10,.0f}")
print(f"{'=' * 70}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f'Monte Carlo FTMO Challenge — Crypto Portfolio — {N_SIMULATIONS:,} Paths',
             fontsize=16, fontweight='bold')

# (0,0) Equity paths
sample_idx = np.random.choice(N_SIMULATIONS, min(200, N_SIMULATIONS), replace=False)
for i in sample_idx:
    c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
    a = 0.4 if (passed[i] or blown[i] or daily_blown[i]) else 0.1
    axes[0, 0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)
axes[0, 0].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2.5,
                   label=f'Target (${FTMO_ACCOUNT*(1+PROFIT_TARGET):,.0f})')
axes[0, 0].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2.5,
                   label=f'Max Loss (${FTMO_ACCOUNT*(1-MAX_TOTAL_LOSS):,.0f})')
axes[0, 0].set_title('Simulated Equity Paths', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Trading Day')
axes[0, 0].set_ylabel('Equity ($)')
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0, 0].legend(loc='upper left', fontsize=9)
axes[0, 0].grid(True, alpha=0.2)

# (0,1) Final equity distribution
bins = np.linspace(final_equity.min(), final_equity.max(), 60)
axes[0, 1].hist(final_equity[passed], bins=bins, color='#2ca02c', alpha=0.7, label=f'Passed ({n_passed:,})')
axes[0, 1].hist(final_equity[blown | daily_blown], bins=bins, color='#d62728', alpha=0.7,
                label=f'Blown ({n_blown_total+n_blown_daily:,})')
axes[0, 1].hist(final_equity[~passed & ~blown & ~daily_blown], bins=bins, color='#aaaaaa', alpha=0.5,
                label=f'Still Trading ({n_survived:,})')
axes[0, 1].axvline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2)
axes[0, 1].axvline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2)
axes[0, 1].set_title('Final Equity Distribution', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Final Equity ($)')
axes[0, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.2)

# (1,0) Outcome pie
labels = ['Passed', 'Blown (Total)', 'Blown (Daily)', 'Still Trading']
sizes = [n_passed, n_blown_total, n_blown_daily, n_survived]
colors_pie = ['#2ca02c', '#d62728', '#ff7f0e', '#aaaaaa']
non_zero = [(l, s, c) for l, s, c in zip(labels, sizes, colors_pie) if s > 0]
if non_zero:
    labels_f, sizes_f, colors_f = zip(*non_zero)
    axes[1, 0].pie(sizes_f, labels=labels_f, colors=colors_f, autopct='%1.1f%%',
                   shadow=True, startangle=90, textprops={'fontsize': 11})
    axes[1, 0].set_title('Outcome Distribution', fontsize=13, fontweight='bold')

# (1,1) Percentile equity curves
for pct, clr in zip([5, 25, 50, 75, 95], ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#17becf']):
    pct_path = np.percentile(equity_paths, pct, axis=0)
    axes[1, 1].plot(range(TRADING_DAYS + 1), pct_path, color=clr, linewidth=2, label=f'P{pct}')
axes[1, 1].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2, alpha=0.7)
axes[1, 1].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2, alpha=0.7)
axes[1, 1].set_title('Percentile Equity Curves', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Trading Day')
axes[1, 1].set_ylabel('Equity ($)')
axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

pass_rate = n_passed / N_SIMULATIONS * 100
print(f"\n{'=' * 70}")
if pass_rate >= 50:
    mc_verdict = f"FAVORABLE -- {pass_rate:.1f}% pass rate"
elif pass_rate >= 25:
    mc_verdict = f"POSSIBLE -- {pass_rate:.1f}% pass rate"
elif pass_rate >= 10:
    mc_verdict = f"CHALLENGING -- {pass_rate:.1f}% pass rate"
else:
    mc_verdict = f"UNLIKELY -- {pass_rate:.1f}% pass rate"
print(f"FTMO VERDICT: {mc_verdict}")
print(f"{'=' * 70}")

In [ ]:
# EXPORT — Summary + CSV + Universal Export

STRATEGY_NAME = "Crypto_Portfolio_FTMO"
TICKER = "PORTFOLIO"

# Build summary dict matching the universal export format
summary = {
    'strategy': STRATEGY_NAME,
    'ticker': 'CRYPTO_PORTFOLIO',
    'components': [f"{s.upper()} {t} ({w:.0%})" for s, t, w, _ in PORTFOLIO],
    'params': {f"{s}_{t}": p for s, t, _, p in PORTFOLIO},
    'full_sample': m_full,
    'in_sample': m_is,
    'out_of_sample': m_oos,
    'buy_hold': m_bh,
    'component_results': component_results,
    'correlation': {
        'avg_pairwise': float(avg_corr),
        'matrix': corr.to_dict(),
    },
    'monte_carlo': {
        'n_simulations': N_SIMULATIONS,
        'pass_rate': float(pass_rate),
        'blown_total_rate': float(n_blown_total / N_SIMULATIONS * 100),
        'blown_daily_rate': float(n_blown_daily / N_SIMULATIONS * 100),
        'median_final_equity': float(np.median(final_equity)),
        'verdict': mc_verdict,
    },
    'ftmo_rules': {
        'account': FTMO_ACCOUNT,
        'profit_target': PROFIT_TARGET,
        'max_daily_loss': MAX_DAILY_LOSS,
        'max_total_loss': MAX_TOTAL_LOSS,
    },
    'data_range': f"{common_start.date()} to {common_end.date()}",
    'train_ratio': TRAIN_RATIO,
    'n_bars': n_bars,
}

import json, os
from datetime import datetime

# Determine export root
if os.path.exists('/content/drive/MyDrive'):
    export_root = '/content/drive/MyDrive/strategy_exports'
else:
    export_root = os.path.join(os.getcwd(), 'strategy_exports')

export_dir = os.path.join(export_root, STRATEGY_NAME, 'PORTFOLIO', 'latest')
os.makedirs(export_dir, exist_ok=True)

# Save summary JSON
with open(os.path.join(export_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

# Save portfolio daily returns
portfolio_daily_returns.to_csv(os.path.join(export_dir, 'daily_returns.csv'))

# Save component results
results_df.to_csv(os.path.join(export_dir, 'component_results.csv'), index=False)

# Save correlation matrix
corr.to_csv(os.path.join(export_dir, 'correlation_matrix.csv'))

print(f"Exported to: {export_dir}")
print(f"  summary.json")
print(f"  daily_returns.csv")
print(f"  component_results.csv")
print(f"  correlation_matrix.csv")

In [ ]:
# TELEGRAM NOTIFICATION (optional — sends results summary to your bot)

import urllib.request
import urllib.parse

TG_BOT_TOKEN = os.getenv('TG_BOT_TOKEN', '8691594427:AAGKbcObikmFxr3yJk5kVkIFkIDAuVyqeoo')
TG_CHAT_ID = os.getenv('TG_CHAT_ID', '6451760231')

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage"
    data = urllib.parse.urlencode({
        'chat_id': TG_CHAT_ID,
        'text': message,
        'parse_mode': 'Markdown',
    }).encode()
    try:
        req = urllib.request.Request(url, data=data)
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status == 200
    except Exception as e:
        print(f"Telegram send failed: {e}")
        return False

# Build results message
msg_lines = [
    '*Crypto Portfolio FTMO -- Backtest Results*\n',
    f"Period: {common_start.date()} to {common_end.date()}",
    f"Components: {len(PORTFOLIO)}\n",
    '*Portfolio Metrics (OOS):*',
    f"  Sharpe: {m_oos['sharpe']:.3f}",
    f"  Return: {m_oos['total_return']:.1%}",
    f"  Max DD: {m_oos['max_dd']:.1%}",
    f"  Calmar: {m_oos['calmar']:.3f}\n",
    '*Full Sample:*',
    f"  Sharpe: {m_full['sharpe']:.3f}",
    f"  Return: {m_full['total_return']:.1%}",
    f"  Max DD: {m_full['max_dd']:.1%}\n",
    '*FTMO Monte Carlo:*',
    f"  Pass Rate: {pass_rate:.1f}%",
    f"  Verdict: {mc_verdict}\n",
    '*Components:*',
]
for r in component_results:
    msg_lines.append(f"  {r['component']}: OOS Sharpe={r['oos_sharpe']:.2f}, WR={r['win_rate']:.0f}%")

msg_lines.append(f"\nAvg Correlation: {avg_corr:.3f}")
msg_lines.append(f"B&H Return: {m_bh['total_return']:.1%}")

msg = '\n'.join(msg_lines)
if send_telegram(msg):
    print("Results sent to Telegram!")
else:
    print("Telegram notification skipped or failed.")
    print("\nMessage preview:")
    print(msg)